# Dataset Logger Module - Interactive Analysis

This notebook provides an interactive interface for:
1. Logging datasets before and after preprocessing
2. Visualizing graphs (KG and user-item interaction graphs) for specific data slices
3. Analyzing data quality and getting recommendations for improving preprocessing

## Setup


In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image
import json
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from collections import Counter

# Set matplotlib to inline
%matplotlib inline

# Try to import networkx for graph visualization
try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False
    print("Warning: networkx not available. Graph visualization will be limited.")

# Configuration
RATING_FILE_NAME = dict({'movie': 'ratings.csv', 'book': 'BX-Book-Ratings.csv', 'music': 'user_artists.dat'})
SEP = dict({'movie': ',', 'book': ';', 'music': '\t'})

print("Setup complete!")


ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

ImportError: numpy._core.multiarray failed to import

## 1. DatasetLogger Class Definition

The DatasetLogger class provides all the functionality for logging and analyzing datasets.


In [ ]:
class DatasetLogger:
    """
    A comprehensive logging and analysis tool for dataset preprocessing.
    """
    
    def __init__(self, dataset_name: str, log_dir: str = '../logs'):
        """Initialize the DatasetLogger."""
        self.dataset_name = dataset_name
        self.log_dir = log_dir
        self.ensure_log_dir()
        
        # Storage for logged data
        self.preprocessing_stats = {
            'before': {},
            'after': {}
        }
        self.timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
    def ensure_log_dir(self):
        """Create log directory if it doesn't exist."""
        os.makedirs(self.log_dir, exist_ok=True)
        os.makedirs(os.path.join(self.log_dir, 'visualizations'), exist_ok=True)
        os.makedirs(os.path.join(self.log_dir, 'statistics'), exist_ok=True)
        os.makedirs(os.path.join(self.log_dir, 'graphs'), exist_ok=True)
    
    def log_raw_data(self, rating_file_path: str, kg_file_path: Optional[str] = None, 
                     item_mapping_file: Optional[str] = None, sep: str = '\t'):
        """Log raw dataset before preprocessing."""
        print(f"Logging raw data for dataset: {self.dataset_name}")
        
        stats = {}
        
        # Log rating data
        try:
            if rating_file_path.endswith('.csv'):
                df_ratings = pd.read_csv(rating_file_path, sep=sep)
            elif rating_file_path.endswith('.dat'):
                df_ratings = pd.read_csv(rating_file_path, sep=sep, header=None)
                if df_ratings.shape[1] >= 3:
                    df_ratings.columns = ['user_id', 'item_id', 'rating'] + \
                                       [f'col_{i}' for i in range(3, df_ratings.shape[1])]
            else:
                df_ratings = pd.read_csv(rating_file_path, sep=sep)
            
            stats['ratings'] = {
                'num_ratings': len(df_ratings),
                'num_unique_users': df_ratings.iloc[:, 0].nunique(),
                'num_unique_items': df_ratings.iloc[:, 1].nunique() if df_ratings.shape[1] > 1 else 0,
                'rating_stats': {
                    'min': float(df_ratings.iloc[:, 2].min()) if df_ratings.shape[1] > 2 else None,
                    'max': float(df_ratings.iloc[:, 2].max()) if df_ratings.shape[1] > 2 else None,
                    'mean': float(df_ratings.iloc[:, 2].mean()) if df_ratings.shape[1] > 2 else None,
                    'median': float(df_ratings.iloc[:, 2].median()) if df_ratings.shape[1] > 2 else None,
                    'std': float(df_ratings.iloc[:, 2].std()) if df_ratings.shape[1] > 2 else None,
                },
                'sparsity': self._calculate_sparsity(
                    df_ratings.iloc[:, 0].nunique(),
                    df_ratings.iloc[:, 1].nunique() if df_ratings.shape[1] > 1 else 0,
                    len(df_ratings)
                )
            }
            
            # Save sample of raw data
            sample_file = os.path.join(self.log_dir, 'statistics', 
                                      f'{self.dataset_name}_raw_ratings_sample.csv')
            df_ratings.head(1000).to_csv(sample_file, index=False)
            
        except Exception as e:
            print(f"Error logging raw rating data: {e}")
            stats['ratings'] = {'error': str(e)}
        
        # Log KG data if available
        if kg_file_path and os.path.exists(kg_file_path):
            try:
                if kg_file_path.endswith('.npy'):
                    kg_data = np.load(kg_file_path)
                else:
                    kg_data = np.loadtxt(kg_file_path, dtype=np.int64)
                
                stats['kg'] = {
                    'num_triples': len(kg_data),
                    'num_unique_entities': len(set(kg_data[:, 0]) | set(kg_data[:, 2])) if kg_data.shape[1] > 2 else 0,
                    'num_unique_relations': len(set(kg_data[:, 1])) if kg_data.shape[1] > 1 else 0,
                }
                
                # Relation frequency distribution
                if kg_data.shape[1] > 1:
                    relation_counts = Counter(kg_data[:, 1])
                    stats['kg']['relation_distribution'] = {
                        str(k): int(v) for k, v in relation_counts.most_common(20)
                    }
                
            except Exception as e:
                print(f"Error logging raw KG data: {e}")
                stats['kg'] = {'error': str(e)}
        
        # Log item mapping if available
        if item_mapping_file and os.path.exists(item_mapping_file):
            try:
                with open(item_mapping_file, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                stats['item_mapping'] = {
                    'num_items': len(lines)
                }
            except Exception as e:
                print(f"Error logging item mapping: {e}")
        
        self.preprocessing_stats['before'] = stats
        
        # Save statistics to JSON
        self._save_stats('before')
        
        print(f"Raw data logged. Statistics saved to {self.log_dir}")
        return stats
    
    def log_processed_data(self, rating_data: np.ndarray, kg_data: Optional[np.ndarray] = None,
                          n_user: Optional[int] = None, n_item: Optional[int] = None,
                          n_entity: Optional[int] = None, n_relation: Optional[int] = None,
                          train_data: Optional[np.ndarray] = None,
                          eval_data: Optional[np.ndarray] = None,
                          test_data: Optional[np.ndarray] = None):
        """Log processed dataset after preprocessing."""
        print(f"Logging processed data for dataset: {self.dataset_name}")
        
        stats = {}
        
        # Log rating data
        if rating_data is not None and len(rating_data) > 0:
            stats['ratings'] = {
                'num_ratings': len(rating_data),
                'num_unique_users': len(set(rating_data[:, 0])) if rating_data.shape[1] > 0 else 0,
                'num_unique_items': len(set(rating_data[:, 1])) if rating_data.shape[1] > 1 else 0,
                'positive_ratings': int(np.sum(rating_data[:, 2] >= 0.5)) if rating_data.shape[1] > 2 else 0,
                'negative_ratings': int(np.sum(rating_data[:, 2] < 0.5)) if rating_data.shape[1] > 2 else 0,
            }
            
            if n_user and n_item:
                stats['ratings']['sparsity'] = self._calculate_sparsity(
                    n_user, n_item, len(rating_data)
                )
        
        # Log KG data
        if kg_data is not None and len(kg_data) > 0:
            stats['kg'] = {
                'num_triples': len(kg_data),
                'num_unique_entities': len(set(kg_data[:, 0]) | set(kg_data[:, 2])) if kg_data.shape[1] > 2 else 0,
                'num_unique_relations': len(set(kg_data[:, 1])) if kg_data.shape[1] > 1 else 0,
            }
            
            # Relation frequency distribution
            if kg_data.shape[1] > 1:
                relation_counts = Counter(kg_data[:, 1])
                stats['kg']['relation_distribution'] = {
                    str(k): int(v) for k, v in relation_counts.most_common(20)
                }
        
        # Log entity/item counts
        if n_user:
            stats['n_user'] = n_user
        if n_item:
            stats['n_item'] = n_item
        if n_entity:
            stats['n_entity'] = n_entity
        if n_relation:
            stats['n_relation'] = n_relation
        
        # Log data splits
        if train_data is not None:
            stats['splits'] = {
                'train_size': len(train_data),
                'train_positive': int(np.sum(train_data[:, 2] >= 0.5)) if train_data.shape[1] > 2 else 0,
            }
        if eval_data is not None:
            if 'splits' not in stats:
                stats['splits'] = {}
            stats['splits']['eval_size'] = len(eval_data)
            stats['splits']['eval_positive'] = int(np.sum(eval_data[:, 2] >= 0.5)) if eval_data.shape[1] > 2 else 0
        if test_data is not None:
            if 'splits' not in stats:
                stats['splits'] = {}
            stats['splits']['test_size'] = len(test_data)
            stats['splits']['test_positive'] = int(np.sum(test_data[:, 2] >= 0.5)) if test_data.shape[1] > 2 else 0
        
        self.preprocessing_stats['after'] = stats
        
        # Save statistics to JSON
        self._save_stats('after')
        
        # Generate comparison report
        self._generate_comparison_report()
        
        print(f"Processed data logged. Statistics saved to {self.log_dir}")
        return stats
    
    def visualize_kg_subgraph(self, kg_data: np.ndarray, entity_ids: Optional[List[int]] = None,
                             max_nodes: int = 100, max_edges: int = 500,
                             output_filename: Optional[str] = None):
        """Visualize a subgraph of the knowledge graph."""
        if not HAS_NETWORKX:
            print("networkx not available. Cannot create graph visualization.")
            return
        
        print(f"Creating KG subgraph visualization for {self.dataset_name}")
        
        # Create graph
        G = nx.MultiDiGraph()
        
        # If entity_ids provided, filter triples containing those entities
        if entity_ids:
            entity_set = set(entity_ids)
            filtered_triples = []
            for triple in kg_data[:max_edges]:
                if triple[0] in entity_set or triple[2] in entity_set:
                    filtered_triples.append(triple)
            triples_to_use = filtered_triples[:max_edges]
        else:
            # Sample random triples
            if len(kg_data) > max_edges:
                indices = np.random.choice(len(kg_data), max_edges, replace=False)
                triples_to_use = kg_data[indices]
            else:
                triples_to_use = kg_data
        
        # Add edges to graph
        for triple in triples_to_use:
            head, relation, tail = triple[0], triple[1], triple[2]
            G.add_edge(head, tail, relation=relation)
        
        # Limit nodes if graph is too large
        if G.number_of_nodes() > max_nodes:
            # Get nodes with highest degree
            degrees = dict(G.degree())
            top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:max_nodes]
            node_ids = [node for node, _ in top_nodes]
            G = G.subgraph(node_ids).copy()
        
        # Create visualization
        plt.figure(figsize=(16, 12))
        
        # Use spring layout for better visualization
        pos = nx.spring_layout(G, k=1, iterations=50)
        
        # Draw nodes
        nx.draw_networkx_nodes(G, pos, node_color='lightblue', 
                              node_size=500, alpha=0.7)
        
        # Draw edges
        nx.draw_networkx_edges(G, pos, edge_color='gray', 
                              arrows=True, arrowsize=10, alpha=0.5, 
                              connectionstyle='arc3,rad=0.1')
        
        # Draw labels (only for a subset to avoid clutter)
        if G.number_of_nodes() <= 50:
            nx.draw_networkx_labels(G, pos, font_size=8)
        
        plt.title(f'Knowledge Graph Subgraph - {self.dataset_name}\n'
                 f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}', 
                 fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        
        # Save figure
        if output_filename is None:
            output_filename = f'{self.dataset_name}_kg_subgraph_{self.timestamp}.png'
        
        output_path = os.path.join(self.log_dir, 'visualizations', output_filename)
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"KG subgraph visualization saved to {output_path}")
        return output_path
    
    def visualize_user_item_graph(self, rating_data: np.ndarray, 
                                  user_ids: Optional[List[int]] = None,
                                  max_nodes: int = 200, max_edges: int = 1000,
                                  output_filename: Optional[str] = None):
        """Visualize user-item interaction graph."""
        if not HAS_NETWORKX:
            print("networkx not available. Cannot create graph visualization.")
            return
        
        print(f"Creating user-item graph visualization for {self.dataset_name}")
        
        # Create bipartite graph
        G = nx.Graph()
        
        # Filter by user_ids if provided
        if user_ids:
            user_set = set(user_ids)
            filtered_ratings = []
            for rating in rating_data[:max_edges]:
                if rating[0] in user_set:
                    filtered_ratings.append(rating)
            ratings_to_use = filtered_ratings[:max_edges]
        else:
            # Sample random ratings
            if len(rating_data) > max_edges:
                indices = np.random.choice(len(rating_data), max_edges, replace=False)
                ratings_to_use = rating_data[indices]
            else:
                ratings_to_use = rating_data
        
        # Add edges (only positive interactions)
        for rating in ratings_to_use:
            user_id, item_id = rating[0], rating[1]
            if len(rating) > 2 and rating[2] >= 0.5:  # Only positive interactions
                G.add_edge(f'U{user_id}', f'I{item_id}')
        
        # Limit nodes if graph is too large
        if G.number_of_nodes() > max_nodes:
            # Get nodes with highest degree
            degrees = dict(G.degree())
            top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:max_nodes]
            node_ids = [node for node, _ in top_nodes]
            G = G.subgraph(node_ids).copy()
        
        # Separate user and item nodes
        user_nodes = [n for n in G.nodes() if n.startswith('U')]
        item_nodes = [n for n in G.nodes() if n.startswith('I')]
        
        # Create visualization
        plt.figure(figsize=(16, 12))
        
        # Use bipartite layout
        pos = nx.spring_layout(G, k=2, iterations=50)
        
        # Draw user nodes
        if user_nodes:
            nx.draw_networkx_nodes(G, pos, nodelist=user_nodes, 
                                  node_color='lightcoral', node_size=300, 
                                  alpha=0.7, label='Users')
        
        # Draw item nodes
        if item_nodes:
            nx.draw_networkx_nodes(G, pos, nodelist=item_nodes, 
                                  node_color='lightblue', node_size=300, 
                                  alpha=0.7, label='Items')
        
        # Draw edges
        nx.draw_networkx_edges(G, pos, edge_color='gray', 
                              alpha=0.3, width=0.5)
        
        # Draw labels (only for a subset)
        if G.number_of_nodes() <= 100:
            labels = {n: n[1:] for n in G.nodes()}  # Remove U/I prefix
            nx.draw_networkx_labels(G, pos, labels, font_size=6)
        
        plt.title(f'User-Item Interaction Graph - {self.dataset_name}\n'
                 f'Users: {len(user_nodes)}, Items: {len(item_nodes)}, '
                 f'Interactions: {G.number_of_edges()}', 
                 fontsize=14, fontweight='bold')
        plt.legend()
        plt.axis('off')
        plt.tight_layout()
        
        # Save figure
        if output_filename is None:
            output_filename = f'{self.dataset_name}_user_item_graph_{self.timestamp}.png'
        
        output_path = os.path.join(self.log_dir, 'visualizations', output_filename)
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"User-item graph visualization saved to {output_path}")
        return output_path
    
    def analyze_data_quality(self, rating_data: Optional[np.ndarray] = None,
                            kg_data: Optional[np.ndarray] = None):
        """Analyze data quality and generate insights."""
        print(f"Analyzing data quality for {self.dataset_name}")
        
        analysis = {
            'dataset': self.dataset_name,
            'timestamp': self.timestamp,
            'metrics': {},
            'recommendations': []
        }
        
        # Analyze ratings
        if rating_data is not None and len(rating_data) > 0:
            n_users = len(set(rating_data[:, 0]))
            n_items = len(set(rating_data[:, 1]))
            n_ratings = len(rating_data)
            
            sparsity = self._calculate_sparsity(n_users, n_items, n_ratings)
            
            # User activity distribution
            user_activity = Counter(rating_data[:, 0])
            item_popularity = Counter(rating_data[:, 1])
            
            analysis['metrics']['ratings'] = {
                'sparsity': sparsity,
                'avg_ratings_per_user': n_ratings / n_users if n_users > 0 else 0,
                'avg_ratings_per_item': n_ratings / n_items if n_items > 0 else 0,
                'user_activity': {
                    'min': min(user_activity.values()) if user_activity else 0,
                    'max': max(user_activity.values()) if user_activity else 0,
                    'median': np.median(list(user_activity.values())) if user_activity else 0,
                },
                'item_popularity': {
                    'min': min(item_popularity.values()) if item_popularity else 0,
                    'max': max(item_popularity.values()) if item_popularity else 0,
                    'median': np.median(list(item_popularity.values())) if item_popularity else 0,
                }
            }
            
            # Recommendations based on metrics
            if sparsity > 0.99:
                analysis['recommendations'].append(
                    "Very high sparsity detected. Consider using collaborative filtering techniques."
                )
            
            if analysis['metrics']['ratings']['avg_ratings_per_user'] < 5:
                analysis['recommendations'].append(
                    "Low average ratings per user. Consider filtering users with very few interactions."
                )
            
            if analysis['metrics']['ratings']['item_popularity']['max'] / \
               max(analysis['metrics']['ratings']['item_popularity']['min'], 1) > 100:
                analysis['recommendations'].append(
                    "High item popularity variance. Consider popularity-based filtering or normalization."
                )
        
        # Analyze KG
        if kg_data is not None and len(kg_data) > 0:
            n_entities = len(set(kg_data[:, 0]) | set(kg_data[:, 2]))
            n_relations = len(set(kg_data[:, 1]))
            n_triples = len(kg_data)
            
            # Entity connectivity
            entity_degrees = Counter(list(kg_data[:, 0]) + list(kg_data[:, 2]))
            
            analysis['metrics']['kg'] = {
                'avg_triples_per_entity': n_triples / n_entities if n_entities > 0 else 0,
                'entities_per_relation': n_entities / n_relations if n_relations > 0 else 0,
                'entity_connectivity': {
                    'min': min(entity_degrees.values()) if entity_degrees else 0,
                    'max': max(entity_degrees.values()) if entity_degrees else 0,
                    'median': np.median(list(entity_degrees.values())) if entity_degrees else 0,
                }
            }
            
            # Check for isolated entities
            isolated_count = sum(1 for deg in entity_degrees.values() if deg == 1)
            if isolated_count > n_entities * 0.1:
                analysis['recommendations'].append(
                    f"High number of isolated entities ({isolated_count}). "
                    "Consider filtering entities with low connectivity."
                )
        
        # Save analysis
        analysis_file = os.path.join(self.log_dir, 'statistics', 
                                    f'{self.dataset_name}_quality_analysis_{self.timestamp}.json')
        with open(analysis_file, 'w') as f:
            json.dump(analysis, f, indent=2)
        
        print(f"Data quality analysis saved to {analysis_file}")
        return analysis
    
    def generate_statistics_report(self):
        """Generate a comprehensive statistics report comparing before and after preprocessing."""
        print(f"Generating statistics report for {self.dataset_name}")
        
        report = {
            'dataset': self.dataset_name,
            'timestamp': self.timestamp,
            'before_preprocessing': self.preprocessing_stats.get('before', {}),
            'after_preprocessing': self.preprocessing_stats.get('after', {}),
            'changes': {}
        }
        
        # Calculate changes
        before_ratings = self.preprocessing_stats.get('before', {}).get('ratings', {})
        after_ratings = self.preprocessing_stats.get('after', {}).get('ratings', {})
        
        if before_ratings and after_ratings:
            report['changes']['ratings'] = {
                'ratings_reduction': before_ratings.get('num_ratings', 0) - after_ratings.get('num_ratings', 0),
                'ratings_reduction_pct': (
                    (before_ratings.get('num_ratings', 0) - after_ratings.get('num_ratings', 0)) / 
                    max(before_ratings.get('num_ratings', 1), 1) * 100
                ),
                'users_reduction': before_ratings.get('num_unique_users', 0) - after_ratings.get('num_unique_users', 0),
                'items_reduction': before_ratings.get('num_unique_items', 0) - after_ratings.get('num_unique_items', 0),
            }
        
        # Save report
        report_file = os.path.join(self.log_dir, 'statistics', 
                                  f'{self.dataset_name}_preprocessing_report_{self.timestamp}.json')
        with open(report_file, 'w') as f:
            json.dump(report, f, indent=2)
        
        print(f"Statistics report saved to {report_file}")
        return report
    
    def _calculate_sparsity(self, n_users: int, n_items: int, n_ratings: int) -> float:
        """Calculate sparsity of the user-item matrix."""
        total_possible = n_users * n_items
        if total_possible == 0:
            return 1.0
        return 1.0 - (n_ratings / total_possible)
    
    def _save_stats(self, stage: str):
        """Save statistics to JSON file."""
        stats_file = os.path.join(self.log_dir, 'statistics', 
                                 f'{self.dataset_name}_{stage}_stats_{self.timestamp}.json')
        
        stats_to_save = {
            'dataset': self.dataset_name,
            'stage': stage,
            'timestamp': self.timestamp,
            'statistics': self.preprocessing_stats[stage]
        }
        
        with open(stats_file, 'w') as f:
            json.dump(stats_to_save, f, indent=2)
    
    def _generate_comparison_report(self):
        """Generate a comparison visualization between before and after preprocessing."""
        if not self.preprocessing_stats.get('before') or not self.preprocessing_stats.get('after'):
            return
        
        # Create comparison plots
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f'Preprocessing Comparison - {self.dataset_name}', 
                     fontsize=16, fontweight='bold')
        
        # Ratings count comparison
        before_ratings = self.preprocessing_stats['before'].get('ratings', {}).get('num_ratings', 0)
        after_ratings = self.preprocessing_stats['after'].get('ratings', {}).get('num_ratings', 0)
        
        axes[0, 0].bar(['Before', 'After'], [before_ratings, after_ratings], 
                      color=['lightcoral', 'lightblue'])
        axes[0, 0].set_title('Number of Ratings')
        axes[0, 0].set_ylabel('Count')
        
        # Users count comparison
        before_users = self.preprocessing_stats['before'].get('ratings', {}).get('num_unique_users', 0)
        after_users = self.preprocessing_stats['after'].get('n_user', 
                   self.preprocessing_stats['after'].get('ratings', {}).get('num_unique_users', 0))
        
        axes[0, 1].bar(['Before', 'After'], [before_users, after_users],
                      color=['lightcoral', 'lightblue'])
        axes[0, 1].set_title('Number of Users')
        axes[0, 1].set_ylabel('Count')
        
        # Items count comparison
        before_items = self.preprocessing_stats['before'].get('ratings', {}).get('num_unique_items', 0)
        after_items = self.preprocessing_stats['after'].get('n_item',
                   self.preprocessing_stats['after'].get('ratings', {}).get('num_unique_items', 0))
        
        axes[1, 0].bar(['Before', 'After'], [before_items, after_items],
                      color=['lightcoral', 'lightblue'])
        axes[1, 0].set_title('Number of Items')
        axes[1, 0].set_ylabel('Count')
        
        # Sparsity comparison
        before_sparsity = self.preprocessing_stats['before'].get('ratings', {}).get('sparsity', 0)
        after_sparsity = self.preprocessing_stats['after'].get('ratings', {}).get('sparsity', 0)
        
        axes[1, 1].bar(['Before', 'After'], [before_sparsity, after_sparsity],
                      color=['lightcoral', 'lightblue'])
        axes[1, 1].set_title('Sparsity')
        axes[1, 1].set_ylabel('Sparsity Ratio')
        axes[1, 1].set_ylim([0, 1])
        
        plt.tight_layout()
        
        # Save figure
        comparison_file = os.path.join(self.log_dir, 'visualizations',
                                      f'{self.dataset_name}_preprocessing_comparison_{self.timestamp}.png')
        plt.savefig(comparison_file, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"Preprocessing comparison visualization saved to {comparison_file}")

print("DatasetLogger class defined!")


## 2. Initialize Logger

Choose your dataset and initialize the logger.


In [ ]:
# Configuration: Change dataset name here
DATASET_NAME = 'book'  # Options: 'book', 'music', 'movie'

# Initialize logger
logger = DatasetLogger(dataset_name=DATASET_NAME, log_dir='../logs')

print(f"Logger initialized for dataset: {DATASET_NAME}")
print(f"Log directory: {logger.log_dir}")


## 2. Log Raw Data (Before Preprocessing)

Log the raw dataset before any preprocessing steps.


In [ ]:
# Define file paths
rating_file = f'../data/{DATASET_NAME}/{RATING_FILE_NAME.get(DATASET_NAME, "ratings.csv")}'
kg_file = f'../data/{DATASET_NAME}/kg.txt'
item_mapping_file = f'../data/{DATASET_NAME}/item_index2entity_id.txt'

# Check if files exist
print("Checking files...")
print(f"Rating file exists: {os.path.exists(rating_file)}")
print(f"KG file exists: {os.path.exists(kg_file)}")
print(f"Item mapping file exists: {os.path.exists(item_mapping_file)}")

# Log raw data
if os.path.exists(rating_file):
    raw_stats = logger.log_raw_data(
        rating_file_path=rating_file,
        kg_file_path=kg_file if os.path.exists(kg_file) else None,
        item_mapping_file=item_mapping_file if os.path.exists(item_mapping_file) else None,
        sep=SEP.get(DATASET_NAME, '\t')
    )
    
    # Display statistics
    print("\n" + "="*60)
    print("RAW DATA STATISTICS")
    print("="*60)
    display(pd.DataFrame([raw_stats]).T)
else:
    print(f"Warning: Raw rating file not found at {rating_file}")


## 3. Log Processed Data (After Preprocessing)

Log the processed dataset after preprocessing.


In [ ]:
# Load processed data
rating_file_processed = f'../data/{DATASET_NAME}/ratings_final'
kg_file_processed = f'../data/{DATASET_NAME}/kg_final'

rating_data = None
kg_data = None

# Load rating data
if os.path.exists(rating_file_processed + '.npy'):
    rating_data = np.load(rating_file_processed + '.npy')
    print(f"Loaded rating data from .npy: {rating_data.shape}")
elif os.path.exists(rating_file_processed + '.txt'):
    rating_data = np.loadtxt(rating_file_processed + '.txt', dtype=np.int64)
    print(f"Loaded rating data from .txt: {rating_data.shape}")
else:
    print(f"Warning: Processed rating file not found at {rating_file_processed}")

# Load KG data
if os.path.exists(kg_file_processed + '.npy'):
    kg_data = np.load(kg_file_processed + '.npy')
    print(f"Loaded KG data from .npy: {kg_data.shape}")
elif os.path.exists(kg_file_processed + '.txt'):
    kg_data = np.loadtxt(kg_file_processed + '.txt', dtype=np.int64)
    print(f"Loaded KG data from .txt: {kg_data.shape}")
else:
    print(f"Warning: Processed KG file not found at {kg_file_processed}")


In [ ]:
# Calculate statistics
if rating_data is not None and len(rating_data) > 0:
    n_user = len(set(rating_data[:, 0]))
    n_item = len(set(rating_data[:, 1]))
    
    n_entity = None
    n_relation = None
    if kg_data is not None and len(kg_data) > 0:
        n_entity = len(set(kg_data[:, 0]) | set(kg_data[:, 2]))
        n_relation = len(set(kg_data[:, 1]))
    
    # Log processed data
    processed_stats = logger.log_processed_data(
        rating_data=rating_data,
        kg_data=kg_data,
        n_user=n_user,
        n_item=n_item,
        n_entity=n_entity,
        n_relation=n_relation
    )
    
    # Display statistics
    print("\n" + "="*60)
    print("PROCESSED DATA STATISTICS")
    print("="*60)
    display(pd.DataFrame([processed_stats]).T)
else:
    print("No processed data to log.")


## 4. Visualize Knowledge Graph Subgraph

Visualize a subgraph of the knowledge graph. You can specify specific entities or let it sample randomly.


In [ ]:
# Configuration for KG visualization
KG_MAX_NODES = 100  # Maximum number of nodes to display
KG_MAX_EDGES = 500  # Maximum number of edges to display
KG_ENTITY_IDS = None  # Set to specific entity IDs list, or None for random sampling
# Example: KG_ENTITY_IDS = [1, 5, 10, 15, 20]

if kg_data is not None and len(kg_data) > 0:
    output_path = logger.visualize_kg_subgraph(
        kg_data=kg_data,
        entity_ids=KG_ENTITY_IDS,
        max_nodes=KG_MAX_NODES,
        max_edges=KG_MAX_EDGES,
        output_filename=f'{DATASET_NAME}_kg_subgraph_interactive.png'
    )
    
    # Display the image inline
    from IPython.display import Image
    display(Image(output_path))
    print(f"\nGraph saved to: {output_path}")
else:
    print("No KG data available for visualization.")


## 5. Visualize User-Item Interaction Graph

Visualize the user-item interaction graph. You can specify specific users or let it sample randomly.


In [ ]:
# Configuration for user-item graph visualization
UI_MAX_NODES = 200  # Maximum number of nodes to display
UI_MAX_EDGES = 1000  # Maximum number of edges to display
UI_USER_IDS = None  # Set to specific user IDs list, or None for random sampling
# Example: UI_USER_IDS = [100, 200, 300]

if rating_data is not None and len(rating_data) > 0:
    output_path = logger.visualize_user_item_graph(
        rating_data=rating_data,
        user_ids=UI_USER_IDS,
        max_nodes=UI_MAX_NODES,
        max_edges=UI_MAX_EDGES,
        output_filename=f'{DATASET_NAME}_user_item_graph_interactive.png'
    )
    
    # Display the image inline
    from IPython.display import Image
    display(Image(output_path))
    print(f"\nGraph saved to: {output_path}")
else:
    print("No rating data available for visualization.")


## 6. Data Quality Analysis

Analyze the quality of your dataset and get recommendations for improving preprocessing.


In [ ]:
# Perform data quality analysis
analysis = logger.analyze_data_quality(
    rating_data=rating_data,
    kg_data=kg_data
)

# Display metrics
print("="*60)
print("DATA QUALITY METRICS")
print("="*60)

if 'metrics' in analysis:
    if 'ratings' in analysis['metrics']:
        print("\nRatings Metrics:")
        ratings_metrics = analysis['metrics']['ratings']
        for key, value in ratings_metrics.items():
            if isinstance(value, dict):
                print(f"  {key}:")
                for k, v in value.items():
                    print(f"    {k}: {v}")
            else:
                print(f"  {key}: {value}")
    
    if 'kg' in analysis['metrics']:
        print("\nKnowledge Graph Metrics:")
        kg_metrics = analysis['metrics']['kg']
        for key, value in kg_metrics.items():
            if isinstance(value, dict):
                print(f"  {key}:")
                for k, v in value.items():
                    print(f"    {k}: {v}")
            else:
                print(f"  {key}: {value}")

# Display recommendations
print("\n" + "="*60)
print("RECOMMENDATIONS FOR IMPROVING PREPROCESSING")
print("="*60)
if analysis.get('recommendations'):
    for i, rec in enumerate(analysis['recommendations'], 1):
        print(f"{i}. {rec}")
else:
    print("No specific recommendations at this time.")


## 7. Generate Statistics Report

Generate a comprehensive report comparing before and after preprocessing.


In [ ]:
# Generate statistics report
report = logger.generate_statistics_report()

# Display report
print("="*60)
print("PREPROCESSING COMPARISON REPORT")
print("="*60)
print(f"\nDataset: {report['dataset']}")
print(f"Timestamp: {report['timestamp']}")

if 'changes' in report and 'ratings' in report['changes']:
    changes = report['changes']['ratings']
    print("\nChanges in Ratings:")
    print(f"  Ratings reduction: {changes.get('ratings_reduction', 0)}")
    print(f"  Ratings reduction (%): {changes.get('ratings_reduction_pct', 0):.2f}%")
    print(f"  Users reduction: {changes.get('users_reduction', 0)}")
    print(f"  Items reduction: {changes.get('items_reduction', 0)}")

# Display comparison visualization if available
comparison_file = f"../logs/visualizations/{DATASET_NAME}_preprocessing_comparison_{logger.timestamp}.png"
if os.path.exists(comparison_file):
    from IPython.display import Image
    print("\nComparison Visualization:")
    display(Image(comparison_file))


## 8. Interactive Data Exploration

Explore specific aspects of your data interactively.


In [ ]:
# Explore rating distribution
if rating_data is not None and len(rating_data) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # User activity distribution
    from collections import Counter
    user_activity = Counter(rating_data[:, 0])
    activity_counts = list(user_activity.values())
    
    axes[0].hist(activity_counts, bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Number of Ratings per User')
    axes[0].set_ylabel('Number of Users')
    axes[0].set_title('User Activity Distribution')
    axes[0].grid(True, alpha=0.3)
    
    # Item popularity distribution
    item_popularity = Counter(rating_data[:, 1])
    popularity_counts = list(item_popularity.values())
    
    axes[1].hist(popularity_counts, bins=50, edgecolor='black', alpha=0.7, color='orange')
    axes[1].set_xlabel('Number of Ratings per Item')
    axes[1].set_ylabel('Number of Items')
    axes[1].set_title('Item Popularity Distribution')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"User Activity Statistics:")
    print(f"  Min ratings per user: {min(activity_counts)}")
    print(f"  Max ratings per user: {max(activity_counts)}")
    print(f"  Mean ratings per user: {np.mean(activity_counts):.2f}")
    print(f"  Median ratings per user: {np.median(activity_counts):.2f}")
    
    print(f"\nItem Popularity Statistics:")
    print(f"  Min ratings per item: {min(popularity_counts)}")
    print(f"  Max ratings per item: {max(popularity_counts)}")
    print(f"  Mean ratings per item: {np.mean(popularity_counts):.2f}")
    print(f"  Median ratings per item: {np.median(popularity_counts):.2f}")


In [ ]:
# Explore KG relation distribution
if kg_data is not None and len(kg_data) > 0:
    from collections import Counter
    
    relation_counts = Counter(kg_data[:, 1])
    
    # Plot top relations
    top_relations = relation_counts.most_common(20)
    relations, counts = zip(*top_relations) if top_relations else ([], [])
    
    if relations:
        plt.figure(figsize=(12, 6))
        plt.barh(range(len(relations)), counts, color='steelblue')
        plt.yticks(range(len(relations)), [f'Relation {r}' for r in relations])
        plt.xlabel('Frequency')
        plt.title('Top 20 Most Frequent Relations in KG')
        plt.gca().invert_yaxis()
        plt.grid(True, alpha=0.3, axis='x')
        plt.tight_layout()
        plt.show()
        
        print(f"Total unique relations: {len(relation_counts)}")
        print(f"Total triples: {len(kg_data)}")
        print(f"\nTop 10 Relations:")
        for i, (rel, count) in enumerate(top_relations[:10], 1):
            print(f"  {i}. Relation {rel}: {count} triples")


## 9. Custom Analysis

Add your own custom analysis code here.


In [ ]:
# Example: Analyze specific user-item interactions
# Customize this cell for your specific analysis needs

if rating_data is not None and len(rating_data) > 0:
    # Example: Find most active users
    from collections import Counter
    user_activity = Counter(rating_data[:, 0])
    
    print("Top 10 Most Active Users:")
    for user_id, count in user_activity.most_common(10):
        print(f"  User {user_id}: {count} interactions")
    
    # Example: Find most popular items
    item_popularity = Counter(rating_data[:, 1])
    
    print("\nTop 10 Most Popular Items:")
    for item_id, count in item_popularity.most_common(10):
        print(f"  Item {item_id}: {count} interactions")
    
    # Add your custom analysis here...


## 10. Export Results

Export analysis results and visualizations.


In [ ]:
# List all generated files
import glob

print("Generated Files:")
print("="*60)

# Statistics files
stats_files = glob.glob(f"../logs/statistics/{DATASET_NAME}*.json")
if stats_files:
    print("\nStatistics Files:")
    for f in sorted(stats_files):
        print(f"  - {f}")

# Visualization files
viz_files = glob.glob(f"../logs/visualizations/{DATASET_NAME}*.png")
if viz_files:
    print("\nVisualization Files:")
    for f in sorted(viz_files):
        print(f"  - {f}")

print(f"\nAll files saved to: {logger.log_dir}")
